When running, make sure to upload chroma index! It is in the chroma folder, make sure to upload both files

In [ ]:
!mkdir -p /content/chroma_db

!mv /content/chroma.sqlite3 /content/chroma_db/

In [ ]:
!unzip /content/97be4841-c863-4115-9200-34c6d225c609-20261011T021757Z-1-001.zip -d /content/chroma_db

Archive:  /content/97be4841-c863-4115-9200-34c6d225c609-20261011T021757Z-1-001.zip
  inflating: /content/chroma_db/97be4841-c863-4115-9200-34c6d225c609/length.bin  
  inflating: /content/chroma_db/97be4841-c863-4115-9200-34c6d225c609/index_metadata.pickle  
  inflating: /content/chroma_db/97be4841-c863-4115-9200-34c6d225c609/header.bin  
  inflating: /content/chroma_db/97be4841-c863-4115-9200-34c6d225c609/link_lists.bin  
  inflating: /content/chroma_db/97be4841-c863-4115-9200-34c6d225c609/data_level0.bin  


In [ ]:
#dependencies

!pip install -q chromadb google-genai google-adk pydantic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 690.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 63.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/135.7 kB 9.5 MB/s eta 0:00:00


In [ ]:
import chromadb
client = chromadb.PersistentClient(path="/content/chroma_db")
collection = client.get_collection("medical_docs")
print(f"Success! Collection has {collection.count()} documents.")

Success! Collection has 2403 documents.


In [ ]:
from google.colab import userdata
import os
os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')

In [ ]:
"""
retrieval_utils.py — FaithfulMed runtime retrieval tools.

Two ADK-ready tools, both reading the persistent ChromaDB MedlinePlus
index built by index_chroma.py:

    retrieve_medical_context(query: str) -> str   # RAG tool
    get_source_context(term: str)  -> str         # Verifier

Design rule: FAIL CLOSED. Any error, empty hit, or weak match returns a
sentinel string instead of a definition.
"""

from __future__ import annotations

import logging
from functools import lru_cache
from typing import Any

import chromadb
from google import genai
from google.genai import types

logger = logging.getLogger(__name__)

# config
CHROMA_PATH       = os.getenv("FAITHFULMED_CHROMA_PATH", "/content/chroma_db")
COLLECTION_NAME   = os.getenv("FAITHFULMED_COLLECTION",  "medical_docs")
EMBED_MODEL       = os.getenv("FAITHFULMED_EMBED_MODEL", "gemini-embedding-2")
EMBED_DIM         = 3072          # MUST match index_chroma.py
EMBED_TASK_QUERY  = "RETRIEVAL_QUERY"   # docs were indexed as RETRIEVAL_DOCUMENT

TOP_K             = 3
MAX_HITS          = 3             # cap on refs handed back
DISTANCE_THRESHOLD = 0.40         # cosine distance; > this => no match   (can adjust later! if its not enough)

NO_MATCH_MSG = "No relevant MedlinePlus definition found."

# singletons?
_genai_client: genai.Client | None = None


def _client() -> genai.Client:
    global _genai_client
    if _genai_client is None:
        _genai_client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
    return _genai_client


@lru_cache(maxsize=1)
def _collection() -> chromadb.Collection:
    chroma = chromadb.PersistentClient(path=CHROMA_PATH)
    return chroma.get_collection(COLLECTION_NAME)

In [ ]:
# embedding - takes query and sends to Gemini to get 3072-dimension vector
# uses api calls
def _embed_query(text: str) -> list[float]:
    resp = _client().models.embed_content(
        model=EMBED_MODEL,
        contents=text,
        config=types.EmbedContentConfig(
            task_type=EMBED_TASK_QUERY,
            output_dimensionality=EMBED_DIM,
        ),
    )
    return list(resp.embeddings[0].values)


# chroma I/O - sends embedded query to chromadb (retrieval!)
# returns dict of documents (text), metadatas (title, source), distances (how close match was)
def _query(query: str) -> list[dict[str, Any]]:
    res = _collection().query(
        query_embeddings=[_embed_query(query)],
        n_results=TOP_K,   # top 3 results
        include=["documents", "metadatas", "distances"],
    )
    docs  = res.get("documents",  [[]])[0]
    metas = res.get("metadatas",  [[]])[0]
    dists = res.get("distances",  [[]])[0]
    return [
        {"document": d, "metadata": m or {}, "distance": float(dist)}
        for d, m, dist in zip(docs, metas, dists)
    ]


# formatting - turns raw chroma into readable string
def _format_hit(hit: dict[str, Any]) -> str:
    meta = hit["metadata"]
    title = meta.get("title") or meta.get("name") or "MedlinePlus Entry"
    source = meta.get("source") or meta.get("url") or "MedlinePlus"
    definition = (hit["document"] or "").strip()
    return (
        "[MedlinePlus Reference]\n"
        f"Title: {title}\n"
        f"Definition: {definition}\n"
        f"Source: {source}"
    )

# catches errors, filters results too far away, caps num of results
# if error, returns empty list
def _good_hits(query: str) -> list[dict[str, Any]]:
    try:
        hits = _query(query)
    except Exception:                       # noqa: BLE001 — fail closed
        logger.exception("Retrieval failed for query=%r", query)
        return []
    good = [h for h in hits if h["distance"] <= DISTANCE_THRESHOLD]
    return good[:MAX_HITS]

In [ ]:
# TOOLS
# simplifier calls to get context for writing text
def retrieve_medical_context(query: str) -> str:
    """Fetch plain-language MedlinePlus definitions for a clinical term.

    Args:
        query: A single medical term or phrase extracted from a clinical
            note, e.g. "acute myocardial infarction" or "metformin".

    Returns:
        A formatted block of MedlinePlus references (title, definition,
        source), or the string "No relevant MedlinePlus definition found."
        if nothing passes the similarity threshold.
    """
    if not query or not query.strip():
        return NO_MATCH_MSG

    hits = _good_hits(query.strip())
    if not hits:
        return NO_MATCH_MSG
    return "\n\n".join(_format_hit(h) for h in hits)

# verifier will call this tool as ground truth
def get_source_context(term: str) -> str:
    """Fetch the RAW MedlinePlus source text for a term, for the Verifier.

    Identical retrieval to retrieve_medical_context, but intended for
    faithfulness checking: the Verifier compares draft claims against
    this text. Returns "No relevant MedlinePlus definition found." when
    the term is ungrounded — the Verifier should treat that as a signal
    that any claim about this term is unsupported.

    Args:
        term: A term, phrase, or claim-keyword to look up verbatim.

    Returns:
        Formatted raw MedlinePlus reference block, or the no-match sentinel.
    """
    if not term or not term.strip():
        return NO_MATCH_MSG

    hits = _good_hits(term.strip())
    if not hits:
        return NO_MATCH_MSG
    # Raw text; title/source included so the Verifier can cite provenance.
    return "\n\n".join(_format_hit(h) for h in hits)

In [ ]:
#test 1 (when distance_threshold was 0.75)
if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO)
    for q in ["acute myocardial infarction", "metformin", "zorblax quarantine"]:
        print(f"\n=== {q} ===")
        print(retrieve_medical_context(q))


=== acute myocardial infarction ===
[MedlinePlus Reference]
Title: Heart Attack
Definition: It can build up for years. Eventually an area of plaque can rupture (break open). A blood clot can form around the plaque and block the artery. A less common cause of heart attack is a severe spasm (tightening) of a coronary artery. The spasm cuts off blood flow through the artery. At the hospital, health care providers make a diagnosis based on your symptoms, blood tests, and different heart health tests . Treatments may include medicines and medical procedures such as coronary angioplasty . After a heart attack, cardiac rehabilitation and lifestyle changes can help you recover. NIH: National Heart, Lung, and Blood Institute
Source: https://medlineplus.gov/heartattack.html

[MedlinePlus Reference]
Title: Heart Attack
Definition: Each year almost 800,000 Americans have a heart attack. A heart attack happens when blood flow to the heart suddenly becomes blocked. Without the blood coming in, the 

In [ ]:
# A mix of terms: exact matches, synonyms, and nonsense
test_queries = [
    "acute myocardial infarction",  # Should match "Heart Attack"
    "metformin",                    # Should match "Metformin"
    "hypertension",                 # Should match "High Blood Pressure"
    "zorblax quarantine",           # Nonsense, should match nothing
    "the quick brown fox"           # Nonsense, should match nothing
]

client = chromadb.PersistentClient(path="/content/chroma_db")
collection = client.get_collection("medical_docs")

print(f"{'Query':<30} | {'Top Match Title':<25} | Distance")
print("-" * 75)

for q in test_queries:
    vec = _embed_query(q)
    res = collection.query(query_embeddings=[vec], n_results=1, include=["metadatas", "distances"])

    if res["distances"][0]:
        dist = res["distances"][0][0]
        title = res["metadatas"][0][0].get("title", "Unknown")
        print(f"{q:<30} | {title:<25} | {dist:.4f}")
    else:
        print(f"{q:<30} | {'NO MATCH':<25} | N/A")

Query                          | Top Match Title           | Distance
---------------------------------------------------------------------------
acute myocardial infarction    | Heart Attack              | 0.2791
metformin                      | Diabetes Medicines        | 0.3451
hypertension                   | How to Prevent High Blood Pressure | 0.2479
zorblax quarantine             | Plague                    | 0.4677
the quick brown fox            | Maximum Heart Rate        | 0.4840


good matches (1-3) are good, but gap of 0.35 between 0.47 (0.12) of worst good match and best bad match is narrow (but usable?)

action: changing distance_threshold from 0.75 to 0.40

In [ ]:
test_queries = [
    # Should match well (baseline sanity)
    "acute myocardial infarction",
    "metformin",
    "hypertension",

    # Real clinical terms, possibly weak/no MedlinePlus coverage:
    "troponin I elevation",
    "left ventricular ejection fraction",
    "procalcitonin",
    "Glasgow Coma Scale",
    "NNRTI regimen",
    "pseudomonas aeruginosa",
    "spontaneous bacterial peritonitis",
    "CHEOPS pain scale",
    "Roux-en-Y gastric bypass",
    "hemoglobin A1c 7.2 percent",
]
print(f"{'Query':<30} | {'Top Match Title':<25} | Distance")
print("-" * 75)

for q in test_queries:
    vec = _embed_query(q)
    res = collection.query(query_embeddings=[vec], n_results=1, include=["metadatas", "distances"])

    if res["distances"][0]:
        dist = res["distances"][0][0]
        title = res["metadatas"][0][0].get("title", "Unknown")
        print(f"{q:<30} | {title:<25} | {dist:.4f}")
    else:
        print(f"{q:<30} | {'NO MATCH':<25} | N/A")


Query                          | Top Match Title           | Distance
---------------------------------------------------------------------------
acute myocardial infarction    | Heart Attack              | 0.2791
metformin                      | Diabetes Medicines        | 0.3451
hypertension                   | How to Prevent High Blood Pressure | 0.2479
troponin I elevation           | Heart Attack              | 0.3743
left ventricular ejection fraction | Heart Health Tests        | 0.3499
procalcitonin                  | Sepsis                    | 0.3815
Glasgow Coma Scale             | Head Injuries             | 0.3221
NNRTI regimen                  | HIV Medicines             | 0.3156
pseudomonas aeruginosa         | Plague                    | 0.3479
spontaneous bacterial peritonitis | Peritoneal Disorders      | 0.3591
CHEOPS pain scale              | Ear Infections            | 0.4002
Roux-en-Y gastric bypass       | Weight Loss Surgery       | 0.3073
hemoglobin A1c 7.2 per

distance threshold chosen empirically (cosine, gemini-embedding-2):

good matches: 0.22-0.38
wrong match: 0.40 (at boundary...)

0.40 splits gap. Dense retrieval right now, handled later by Verifier?

In [ ]:
#test 2 (when distance_threshold updated to 0.40)

if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO)

    test_terms = {
        "— common conditions (should pass cleanly)": [
            "acute myocardial infarction",
            "hypertension",
            "type 2 diabetes",
            "chronic obstructive pulmonary disease",
            "atrial fibrillation",
            "congestive heart failure",
        ],

        "— common medications (should pass cleanly)": [
            "metformin",
            "lisinopril",
            "atorvastatin",
            "levothyroxine",
            "omeprazole",
            "amoxicillin",
        ],

        "— labs / measurements (mixed coverage)": [
            "hemoglobin A1c",
            "troponin I elevation",
            "creatinine clearance",
            "left ventricular ejection fraction",
            "INR",
            "BUN",
        ],

        "— procedures (mixed coverage)": [
            "coronary artery bypass graft",
            "Roux-en-Y gastric bypass",
            "colonoscopy",
            "cholecystectomy",
            "percutaneous coronary intervention",
        ],

        "— clinical jargon that SHOULD simplify": [
            "dyspnea on exertion",
            "bilateral lower extremity edema",
            "non-ST elevation myocardial infarction",
            "hepatomegaly",
            "tachycardia",
        ],

        "— real clinical concepts with indirect MedlinePlus coverage": [
            "CHEOPS pain scale",
            "pseudomonas aeruginosa",
            "NNRTI regimen",
            "spontaneous bacterial peritonitis",
            "Glasgow Coma Scale score of 12",
        ],

        "— edge cases / boundary probes": [
            "myocardial infarction",
            "heart attack",
            "MI",
            "elevated troponin",
            "chest pain radiating to left arm",
        ],

        "— pure nonsense (sanity check)": [
            "zorblax quarantine",
            "the quick brown fox",
            "flurble gribble syndrome",
        ],
    }

    for category, terms in test_terms.items():
        print(f"\n{'=' * 70}")
        print(category)
        print('=' * 70)
        for q in terms:
            print(f"\n--- {q} ---")
            print(retrieve_medical_context(q))


— common conditions (should pass cleanly)

--- acute myocardial infarction ---
[MedlinePlus Reference]
Title: Heart Attack
Definition: It can build up for years. Eventually an area of plaque can rupture (break open). A blood clot can form around the plaque and block the artery. A less common cause of heart attack is a severe spasm (tightening) of a coronary artery. The spasm cuts off blood flow through the artery. At the hospital, health care providers make a diagnosis based on your symptoms, blood tests, and different heart health tests . Treatments may include medicines and medical procedures such as coronary angioplasty . After a heart attack, cardiac rehabilitation and lifestyle changes can help you recover. NIH: National Heart, Lung, and Blood Institute
Source: https://medlineplus.gov/heartattack.html

[MedlinePlus Reference]
Title: Heart Attack
Definition: Each year almost 800,000 Americans have a heart attack. A heart attack happens when blood flow to the heart suddenly becomes

need to add abbreviation expanion in Extractor before it calls retrieve_medical_context (or will cause false negatives)

for nonsense terms, it sometimes fails closed, and sometimes matches.
1 false positive for pseudomonas (4 out of 5 behaved right - failing closed when it should, or matching right related concept)

right now, most key terms are giving **different** paragraphs from the **same** article as top 3 results. so, should I keep as-is for more depth, or have more diversity filtering (retrieve top-10, keep each unique source/title) for more breadth?